# 04 - Ablation Studies
Position, PCA dim, temperature, and KD-weight ablations.

**Saved outputs:** this notebook writes fold CSVs for position, PCA, temperature, and alpha ablations under `../outputs/tables/`, then rebuilds the corresponding JSON summaries for figure generation.


In [ ]:
import sys; sys.path.insert(0, "..")
from src.analysis.aggregate_results import save_available_ablation_summaries
from src.utils.seed import set_seed
from src.trainers.train_teacher import train_teacher_cv
from src.trainers.train_student import run_student_cv

set_seed(42)
CSV = "../data/raw/wdbc.csv"
teacher = train_teacher_cv(CSV, pca_dim=4)

# Position ablation
for pos in ["front", "middle", "tail"]:
    run_student_cv(CSV, teacher_fold_outputs=teacher, model_type="hybrid",
                   use_kd=True, quantum_position=pos, exp_name=f"ablation_pos_{pos}")

# PCA dim ablation
for dim in [4, 6, 8]:
    t = train_teacher_cv(CSV, pca_dim=dim)
    run_student_cv(CSV, teacher_fold_outputs=t, model_type="hybrid",
                   use_kd=True, pca_dim=dim, exp_name=f"ablation_pca_{dim}")

# Temperature ablation
for T in [2.0, 4.0]:
    run_student_cv(CSV, teacher_fold_outputs=teacher, model_type="hybrid",
                   use_kd=True, T=T, exp_name=f"ablation_T_{T}")

# KD weight ablation (tail position)
for alpha in [0.3, 0.5, 0.7]:
    run_student_cv(CSV, teacher_fold_outputs=teacher, model_type="hybrid",
                   use_kd=True, alpha=alpha, T=2.0, quantum_position="tail",
                   exp_name=f"ablation_alpha_{alpha}")

save_available_ablation_summaries()
